# PRIDE Archive — Proteomics Data Ingestion and Analysis

**PRIDE** (PRoteomics IDEntifications) is the world's largest public repository of mass spectrometry-based proteomics data. It stores raw spectra files, search results, and peptide/protein identifications from thousands of experiments.

Key data types:
| Data type | Description |
|---|---|
| **Projects** | Experimental submissions, each with an accession like `PXD000001` |
| **Files** | Raw spectra (`.raw`, `.mzML`), search results (`.mzIdentML`, `.dat`), peak lists |
| **Peptide identifications** | Sequences, charge states, modifications, scores |
| **Protein identifications** | Inferred proteins with coverage and supporting PSMs |
| **PTMs** | Post-translational modifications detected across experiments |

**Reference:** Perez-Riverol et al. (2022), *Nucleic Acids Research*, PRIDE database and tools

**API base:** `https://www.ebi.ac.uk/pride/ws/archive/v2`

# TODO

* [x] **Ingest data**
    * [x] Connect to PRIDE REST API and confirm access
    * [x] Page through the projects endpoint and download project-level metadata
    * [x] Parse metadata into a Polars DataFrame with correct dtypes
    * [x] Fetch file listings for a project of interest (e.g. a human phosphoproteomics study)
    * [x] Download peptide and protein identification summaries via the API
* [ ] **Explore and clean**
    * [ ] Summarize project counts by organism, instrument, experiment type, and year
    * [ ] Inspect PTM frequency across experiments
    * [ ] Handle missing/null fields in project metadata
* [ ] **Analysis**
    * [ ] Identify the most-studied organisms, diseases, and tissue types in PRIDE
    * [ ] Analyse instrument and software usage trends over time
    * [ ] Cluster projects by keyword/PTM profile (TF-IDF + dimensionality reduction)
* [ ] **Peptide-level deep dive**
    * [ ] For a selected project, load peptide identifications and score distributions
    * [ ] Plot score vs. FDR threshold and discuss target-decoy competition (TDC) approach
    * [ ] Examine modified peptide frequencies and localisation scores
* [ ] **Visualization**
    * [ ] Bar/treemap of submissions by organism and year
    * [ ] Heatmap of instrument × experiment type usage
    * [ ] Score distribution plots for peptide identifications
* [ ] **Statistical analysis**
    * [ ] Explain the target-decoy FDR framework and its mathematical basis
    * [ ] Discuss multiple testing correction at PSM, peptide, and protein levels

In [ ]:
import requests
import time
import json
from pathlib import Path

import polars as pl

## 1. Ingest Data

### 1.1 Connect to PRIDE API and Confirm Access

In [ ]:
PRIDE_BASE = "https://www.ebi.ac.uk/pride/ws/archive/v2"
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

def pride_get(endpoint: str, params: dict = None) -> dict:
    """
    Send a GET request to the PRIDE REST API.

    Parameters
    ----------
    endpoint : str
        API path relative to PRIDE_BASE (e.g. "projects").
    params : dict, optional
        Query parameters.

    Returns
    -------
    dict
        Parsed JSON response.
    """
    url = f"{PRIDE_BASE}/{endpoint}"
    resp = requests.get(url, params=params or {}, timeout=30)
    resp.raise_for_status()
    return resp.json()

# Connectivity check: fetch a single project to confirm the API is reachable
sample = pride_get("projects/PXD000001")
print(f"Title : {sample['title']}")
print(f"Submitted : {sample['submissionDate']}")
print(f"Organism : {[o['name'] for o in sample.get('organisms', [])]}")
print(f"Instruments: {[i['name'] for i in sample.get('instruments', [])]}")

### 1.2 Page Through Projects and Download Metadata

In [ ]:
PROJECTS_CACHE = DATA_DIR / "pride_projects.json"
PAGE_SIZE = 100   # maximum page size supported by the API

def fetch_all_projects(cache_path: Path = PROJECTS_CACHE) -> list[dict]:
    """
    Fetch all public PRIDE projects via the paginated /projects endpoint.
    Results are cached to disk so subsequent runs are instant.

    Parameters
    ----------
    cache_path : Path
        Where to write/read the cached JSON.

    Returns
    -------
    list[dict]
        One dict per project with all metadata fields.
    """
    if cache_path.exists():
        print(f"Loading from cache: {cache_path}")
        return json.loads(cache_path.read_text())

    all_projects = []
    page = 0
    while True:
        resp = pride_get("projects", {"pageSize": PAGE_SIZE, "page": page})
        # The API wraps results in a '_embedded' key when results exist
        batch = resp.get("_embedded", {}).get("projects", [])
        if not batch:
            break
        all_projects.extend(batch)
        total = resp.get("page", {}).get("totalElements", "?")
        print(f"  Page {page:>4} — {len(all_projects):>6} / {total} projects", end="\r")
        page += 1
        time.sleep(0.2)  # be polite to the EBI servers

    print(f"\nDone. Fetched {len(all_projects)} projects.")
    cache_path.write_text(json.dumps(all_projects))
    return all_projects

projects_raw = fetch_all_projects()
print(f"Total projects: {len(projects_raw)}")

### 1.3 Parse into a Polars DataFrame

In [ ]:
def flatten_project(p: dict) -> dict:
    """
    Flatten a single raw project dict into a row-friendly format.
    Nested list fields (organisms, instruments, etc.) are joined as
    pipe-separated strings so they fit in a single DataFrame column.

    Parameters
    ----------
    p : dict
        Raw project record from the PRIDE API.

    Returns
    -------
    dict
        Flat dict with scalar and string-encoded list values.
    """
    def names(lst, key="name"):
        # Extract 'name' (or another key) from a list of dicts; return pipe-joined string
        return " | ".join(item[key] for item in (lst or []) if key in item) or None

    return {
        "accession":          p.get("accession"),
        "title":              p.get("title"),
        "submission_date":    p.get("submissionDate"),
        "publication_date":   p.get("publicationDate"),
        "submission_type":    p.get("submissionType"),
        "organisms":          names(p.get("organisms")),
        "organism_parts":     names(p.get("organismParts")),
        "diseases":           names(p.get("diseases")),
        "instruments":        names(p.get("instruments")),
        "experiment_types":   names(p.get("experimentTypes")),
        "quant_methods":      names(p.get("quantificationMethods")),
        "ptms":               " | ".join(p.get("identifiedPTMStrings") or []) or None,
        "keywords":           " | ".join(p.get("keywords") or []) or None,
        "countries":          " | ".join(p.get("countries") or []) or None,
        "total_downloads":    p.get("totalFileDownloads"),
    }

rows = [flatten_project(p) for p in projects_raw]
projects = pl.DataFrame(rows).with_columns([
    pl.col("submission_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("publication_date").str.to_date("%Y-%m-%d", strict=False),
    pl.col("total_downloads").cast(pl.Int64, strict=False),
])

print(f"Shape: {projects.shape}")
print(f"Memory: {projects.estimated_size('mb'):.1f} MB")
projects.head(5)

### 1.4 Fetch File Listings for a Project of Interest

In [ ]:
# PXD000561: a well-known human phosphoproteomics dataset (Sharma et al., 2014 — deep human proteome)
FOCUS_PROJECT = "PXD000561"

def fetch_project_files(accession: str) -> pl.DataFrame:
    """
    Fetch the file listing for a PRIDE project.

    Parameters
    ----------
    accession : str
        PRIDE project accession, e.g. "PXD000561".

    Returns
    -------
    pl.DataFrame
        One row per file with name, size, type, and download URL.
    """
    resp = pride_get(f"files/byProject", {"accession": accession, "pageSize": 200})
    files = resp.get("_embedded", {}).get("files", [])
    rows = [
        {
            "file_name":   f.get("fileName"),
            "file_type":   f.get("fileCategory", {}).get("name"),
            "size_bytes":  f.get("fileSizeBytes"),
            "download_url": next(
                (l["href"] for l in f.get("_links", {}).values() if "ftp" in l.get("href", "")),
                None,
            ),
        }
        for f in files
    ]
    return pl.DataFrame(rows).with_columns(
        pl.col("size_bytes").cast(pl.Int64, strict=False)
    )

project_files = fetch_project_files(FOCUS_PROJECT)
print(f"Files in {FOCUS_PROJECT}: {len(project_files)}")
print(project_files.group_by("file_type").agg(
    pl.len().alias("count"),
    (pl.col("size_bytes").sum() / 1e9).round(2).alias("total_gb")
).sort("total_gb", descending=True))
project_files.head(10)

### 1.5 Fetch Peptide and Protein Identification Summaries

In [ ]:
def fetch_peptides(accession: str, page_size: int = 100, max_pages: int = 10) -> pl.DataFrame:
    """
    Fetch peptide identifications for a PRIDE project.

    Parameters
    ----------
    accession : str
        PRIDE project accession.
    page_size : int
        Records per API page.
    max_pages : int
        Cap on pages to fetch (to avoid very long downloads during exploration).

    Returns
    -------
    pl.DataFrame
        Peptide sequences with modifications, charge, and best search engine score.
    """
    rows = []
    for page in range(max_pages):
        resp = pride_get(
            "peptideevidences",
            {"projectAccession": accession, "pageSize": page_size, "page": page},
        )
        batch = resp.get("_embedded", {}).get("peptideevidences", [])
        if not batch:
            break
        for pep in batch:
            rows.append({
                "peptide_sequence":   pep.get("peptideSequence"),
                "protein_accession":  pep.get("proteinAccession"),
                "charge":             pep.get("charge"),
                "missed_cleavages":   pep.get("missedCleavages"),
                "best_search_engine_score": next(
                    iter(pep.get("bestSearchEngineScore", {}).values()), None
                ),
                "ptms": " | ".join(
                    m.get("name", "") for m in pep.get("ptms") or []
                ) or None,
            })
        time.sleep(0.2)

    return pl.DataFrame(rows).with_columns(
        pl.col("charge").cast(pl.Int32, strict=False),
        pl.col("missed_cleavages").cast(pl.Int32, strict=False),
        pl.col("best_search_engine_score").cast(pl.Float64, strict=False),
    )

peptides = fetch_peptides(FOCUS_PROJECT)
print(f"Peptide identifications fetched: {len(peptides)}")
peptides.head(10)